In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from cmdstanpy import CmdStanModel
import os

# 1. 데이터 로드
df_spk = pd.read_csv('./speaking/simulated_speaking_data.csv')

# 집단별 사전 총점 추출 (20점 만점)
exp_pre = df_spk[df_spk['Group'] == 'Experimental']['Spk_Pre_Total'].values
ctrl_pre = df_spk[df_spk['Group'] == 'Control']['Spk_Pre_Total'].values

print(f"📊 [말하기 사전 기술통계] 실험집단: {np.mean(exp_pre):.2f} (SD: {np.std(exp_pre):.2f})")
print(f"📊 [말하기 사전 기술통계] 통제집단: {np.mean(ctrl_pre):.2f} (SD: {np.std(ctrl_pre):.2f})")

In [ ]:
# ==========================================
# 2. 고전 통계학: 독립표본 t-검정 (Frequentist)
# ==========================================
t_stat, p_val = stats.ttest_ind(exp_pre, ctrl_pre, equal_var=True)
is_homogeneous = p_val >= 0.05

print("\n--- [고전 통계학 결과] ---")
print(f"t-statistic: {t_stat:.4f}, p-value: {p_val:.4f}")
print(f"결과: {'두 집단은 통계적으로 동질함' if is_homogeneous else '집단 간 유의한 차이 발견'}")

In [ ]:
# ==========================================
# 3. 베이지안 통계학: 모수 추정 (Bayesian)
# ==========================================
stan_code = """
data {
    int<lower=0> n1; int<lower=0> n2;
    vector[n1] y1; vector[n2] y2;
}
parameters {
    real mu1; real mu2;
    real<lower=0> sigma;
}
model {
    mu1 ~ normal(10, 5); // 중급 수준(10점 내외) 반영한 약한 정보적 사전분포
    mu2 ~ normal(10, 5);
    sigma ~ exponential(0.1);
    y1 ~ normal(mu1, sigma);
    y2 ~ normal(mu2, sigma);
}
generated quantities {
    real delta = mu1 - mu2;
}
"""

with open("spk_homogeneity.stan", "w") as f:
    f.write(stan_code)

model = CmdStanModel(stan_file="spk_homogeneity.stan")
stan_data = {'n1': len(exp_pre), 'n2': len(ctrl_pre), 'y1': exp_pre, 'y2': ctrl_pre}
fit = model.sample(data=stan_data, iter_sampling=2000, show_progress=False, seed=42)

# 사후분포 분석
delta_samples = fit.stan_variable('delta')
ci_lo, ci_hi = np.percentile(delta_samples, [2.5, 97.5])
post_mean = np.mean(delta_samples)
contains_zero = ci_lo <= 0 <= ci_hi



print("\n--- [베이지안 통계학 결과] ---")
print(f"평균 차이(delta) 사후 평균: {post_mean:.4f}")
print(f"95% 신용구간 (HDI): [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"결론: {'0을 포함하므로 동질적임' if contains_zero else '0을 포함하지 않으므로 이질적임'}")

In [ ]:
# ==========================================
# 4. 시각화 (1번 파일 스타일 재현)
# ==========================================
plt.figure(figsize=(10, 5))

# delta의 사후분포 히스토그램
plt.hist(delta_samples, bins=50, color='skyblue', alpha=0.7, density=True, label='Posterior of Delta')
plt.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero (No Difference)')
plt.hlines(y=0, xmin=ci_lo, xmax=ci_hi, color='black', linewidth=6, label='95% Credible Interval')

plt.title("Homogeneity Check: Speaking Pre-test (Bayesian)", fontsize=13)
plt.xlabel("Difference in Means (Experimental - Control)")
plt.ylabel("Density")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()